In [1]:
# Import Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#import math
from datetime import datetime, timedelta
#from scipy.stats import loguniform, randint, uniform, skew
#import statsmodels.api as sm
#from sklearn.linear_model import LogisticRegression
#from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_s
#from sklearn.model_selection import cross_val_score, StratifiedKFold, Randomize
#from sklearn.preprocessing import StandardScaler
#from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingCl

#import optuna
#import shap

# Set Display Preferences
pd.set_option('display.max_columns', None)
pd.options.display.float_format = "{:.2f}".format

np.random.seed(1)

In [2]:
### UK Based ecom retailer sales from Jan 12, 2009 to Sep 12, 2011.
### Mainly sells unique all-occasion gift-ware.
aisles = pd.read_csv("data/raw/aisles.csv")
departments = pd.read_csv("data/raw/departments.csv")
order_products__prior = pd.read_csv("data/raw/order_products__prior.csv")
    # contains the products for each past order & then the test & train datasets are for the customers final order
    # len = 32.4M rows (train len only 1.4M & doesnt include eval_set = test)
    # Neither order_products_x contain eval_set = test
    # Does reordered mean that customer has purchased that product in the past. if first purchase, reorder still = 0 instead of null
    # The goal of the original sompetition was to predict which previously purchased products will be in a user’s next order.
order_products__train = pd.read_csv("data/raw/order_products__train.csv")
    # len = 1.38M rows
    # The goal of the original competition was to predict which previously purchased products will be in a user’s next order.
        # so customer i has purchased x different products. objectice is to calc/predict which of those x items shows up in their final order.
        # achieved by calculating a proability (forecast?) for each previous item & if prob > a threshold then predict it will be in final order
            # note: so do we not care about the new items in the final order?????
orders = pd.read_csv("data/raw/orders.csv")
    # contains all orders by all customers in but the prior, train & test sets
    # a customers last order is categorized into train or test
    # a customers first order (order_num=1) has a blank days_since_prior_order which makes sense. sometimes users make multiple orders in 1 day making value of this factor = 0.
        # note: there are a lot of same day orders -> probably dont repeate same products in 1 day
    # max days since reorder is 30 days so 30 actually = 30+
products = pd.read_csv("data/raw/products.csv")


In [3]:
order_products = pd.concat([order_products__prior, order_products__train], ignore_index=True)
df = orders.merge(order_products, how='inner', on='order_id')
df = df.merge(products, how='left', on='product_id')
df = df.merge(aisles, how='left', on='aisle_id')
df = df.merge(departments, how='left', on='department_id')



In [4]:
# Department -> product_cat  (beverages intentionally omitted; handled by aisle below)
dept_to_cat = {
    'produce':         'produce',
    'dairy eggs':      'dairy eggs',
    'meat seafood':    'meat seafood', 
    'snacks':          'indulgence',          
    'dry goods pasta': 'staples',
    'canned goods':    'staples',
    'pantry':          'staples',
    'breakfast':       'staples',
    'bulk':            'staples',
    'deli':            'prepared',
    'bakery':          'prepared',
    'frozen':          'prepared',
    'household':       'non_consumables',
    'personal care':   'non_consumables',
    # standalone markers (kept distinct so they can be exluded from CLR block)
    'alcohol':         'alcohol',
    'babies':          'babies',
    'pets':            'pets',
    'international':   'international',
    # drop
    'other':           'other',
    'missing':         'other',}

# Beverage aisle -> product_cat
bev_to_cat = {
    'water seltzer sparkling water': 'staples',
    'refrigerated':                  'staples',
    'juice nectars':                 'staples',
    'tea':                           'staples',
    'coffee':                        'staples',
    'soft drinks':                   'indulgence',
    'energy sports drinks':          'indulgence',
    'cocoa drink mixes':             'indulgence',}

# Map department (beverages -> NaN here)
df['product_cat'] = df['department'].map(dept_to_cat)
# Map beverages by aisle
bev = df['department'] == 'beverages'
df.loc[bev, 'product_cat'] = df.loc[bev, 'aisle'].map(bev_to_cat)

# Safety check — print unmapped departments/ailes (check for NaN)
print(df['product_cat'].value_counts(dropna=False))

product_cat
produce            9888378
staples            7051743
dairy eggs         5631067
prepared           4657579
indulgence         3510861
non_consumables    1243345
meat seafood        739238
babies              438743
international       281155
alcohol             159294
other               115482
pets                102221
Name: count, dtype: int64


In [5]:
core_cats   = ['produce', 'dairy eggs', 'meat seafood','indulgence', 'staples', 'prepared', 'non_consumables']
marker_cats = ['alcohol', 'babies', 'pets', 'international']

# Transpose item counts by category to a new df
cat_counts = (df[df['product_cat'] != 'other']
              .groupby(['user_id', 'product_cat']).size()
              .unstack(fill_value=0))
for c in core_cats + marker_cats:                                   # ensure all columns exist
    if c not in cat_counts.columns:
        cat_counts[c] = 0

# CLR block — core categories only
    # Re-expresses each category as its log-representation relative to the customer's own mean basket, 
    # Mapping the composition into unconstrained Euclidean space where distances behave and the values stay interpretable 
    # Note: positive = that customer over-indexes on this category relative to their own mix)
    # Note: fixes sum-to-1 constraint that created spurious dependence + values now live in a flat Euclidean space rahter than a constrained simplex
alpha = 0.5                                                         # additive smoothing -> no zeros
core_smoothed = cat_counts[core_cats].astype(float) + alpha
core_comp = core_smoothed.div(core_smoothed.sum(axis=1), axis=0)    # divide by row total to get smoothed category percentages
log_comp = np.log(core_comp)                                        # take log

clr = log_comp.sub(log_comp.mean(axis=1), axis=0)                   # subtract mean of row from each value in that row
clr.columns = [f'clr_{c}' for c in core_cats]                       # asign column names


In [6]:
df



,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department,product_cat
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7,soft drinks,beverages,indulgence
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16,soy lactosefree,dairy eggs,dairy eggs
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19,popcorn jerky,snacks,indulgence
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19,popcorn jerky,snacks,indulgence
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17,paper goods,household,non_consumables
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33819101,272231,206209,train,14,6,14,30.00,40603,4,0,Fabric Softener Sheets,75,17,laundry,household,non_consumables
33819102,272231,206209,train,14,6,14,30.00,15655,5,0,Dark Chocolate Mint Snacking Chocolate,45,19,candy chocolate,snacks,indulgence
33819103,272231,206209,train,14,6,14,30.00,42606,6,0,Phish Food Frozen Yogurt,37,1,ice cream ice,frozen,prepared
33819104,272231,206209,train,14,6,14,30.00,37966,7,0,French Baguette Bread,112,3,bread,bakery,prepared


# 3) Markers — binary "ever reordered from this category" flag per customer
all_users = cat_counts.index   # full user set (same index as clr)

marker_reorders = df[(df['reordered'] > 0) & (df['product_cat'].isin(marker_cats))]

flags = (marker_reorders
         .groupby(['user_id', 'product_cat']).size()   # reordered-item count per user/cat
         .unstack(fill_value=0)
         .gt(0).astype(int)                             # -> 1 if ever reordered, else 0
         .reindex(index=all_users, columns=marker_cats, fill_value=0))  # all users + cats, 0-fill
flags.columns = [f'mk_{c}_reordered' for c in marker_cats]

# sanity check: these sums should match customers_reordered in your y.csv
# (alcohol 9007, babies 19358, pets 8057, international 29510)
print(flags.sum())

# 4) Combine (join behavioural features here too, then standardize all together)
user_features = clr.join(flags)
user_features

In [ ]:
# just some eda on pct of cx reorder by category
y = df.groupby(['product_cat'], as_index=True).agg(
                                            customers_purchased=('user_id', 'nunique'),
                                            customers_reordered = ('user_id', lambda x: x[df.loc[x.index, 'reordered'] > 0].nunique()))

y['customers_purchased_pct'] = y['customers_purchased'] / df['user_id'].nunique() * 100
y['customers_reordered_pct'] = y['customers_reordered'] / df['user_id'].nunique() * 100
y


,customers_purchased,customers_reordered,customers_purchased_pct,customers_reordered_pct
product_cat,,,,
alcohol,16104,9007,7.81,4.37
babies,34782,19358,16.87,9.39
dairy eggs,191861,165703,93.04,80.36
indulgence,183036,136180,88.76,66.04
international,79357,29510,38.48,14.31
meat seafood,116851,70181,56.67,34.03
non_consumables,140269,67532,68.02,32.75
other,44512,15069,21.59,7.31
pets,15484,8057,7.51,3.91


In [8]:
# One-hot encode day of week column
#df['order_day'] = df['order_dow']+1
#df = pd.get_dummies(df, columns=['order_day'], dtype=int)

# One-hot encode product_cat
#df = pd.get_dummies(df, columns=['product_cat'], dtype=int)
# Capture names of all dummy columns (useful later for groupby)
#dummy_cols = [c for c in df.columns if c.startswith('product_cat_')]# or c.startswith('order_day_')]
#dummy_cols.remove('department_id')

# Create a dummy variable for organic purchases 
df['organic'] = df['product_name'].str.contains('organic', case=False).astype(int)

df

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department,product_cat,organic
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7,soft drinks,beverages,indulgence,0
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16,soy lactosefree,dairy eggs,dairy eggs,1
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19,popcorn jerky,snacks,indulgence,0
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19,popcorn jerky,snacks,indulgence,0
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17,paper goods,household,non_consumables,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33819101,272231,206209,train,14,6,14,30.00,40603,4,0,Fabric Softener Sheets,75,17,laundry,household,non_consumables,0
33819102,272231,206209,train,14,6,14,30.00,15655,5,0,Dark Chocolate Mint Snacking Chocolate,45,19,candy chocolate,snacks,indulgence,0
33819103,272231,206209,train,14,6,14,30.00,42606,6,0,Phish Food Frozen Yogurt,37,1,ice cream ice,frozen,prepared,0
33819104,272231,206209,train,14,6,14,30.00,37966,7,0,French Baguette Bread,112,3,bread,bakery,prepared,0


In [ ]:
# Calc avg basket size & std dev of avg basket size
basket_sizes = df.groupby(['user_id', 'order_number'], as_index=False).agg(
                                            basket_size=('product_id', 'nunique'),
                                            aisles_breadth=('aisle_id', 'nunique'),
                                            department_breadth=('department_id', 'nunique'))
#basket_sizes = df.groupby(['user_id', 'order_number']).size().reset_index(name='basket_size')

basket_comp = basket_sizes.groupby('user_id').agg(
                                            avg_basket_size=('basket_size', 'mean'),
                                            std_basket_size=('basket_size', 'std'),
                                            avg_aisles_breadth=('aisles_breadth', 'mean'),
                                            std_aisles_breadth=('aisles_breadth', 'std'),
                                            avg_department_breadth=('department_breadth', 'mean'),
                                            std_department_breadth=('department_breadth', 'std')
                                            )
basket_comp

In [20]:
############################### IM MAKING IT FASTER
df[df['order_dow'].le(1)]

df['is_weekend']   = df['order_dow'].le(1)
df['is_morning']   = df['order_hour_of_day'].between(5, 11)
df['is_afternoon'] = df['order_hour_of_day'].between(12, 19)
df['is_night']     = ~df['order_hour_of_day'].between(5, 19)

# will need to loop for the other markers
df['alcohol_reord'] = (df['product_cat'] == 'alcohol') & (df['reordered'] > 0)


# group by
df_output = df.groupby('user_id').agg(
    orders               = ('order_number', 'nunique'),
    max_order_number     = ('order_number', 'max'),
    unique_days          = ('order_dow', 'nunique'),
    weekend_orders_pct   = ('is_weekend',   'mean'),
    morning_orders_pct   = ('is_morning',   'mean'),
    afternoon_orders_pct = ('is_afternoon', 'mean'),
    night_orders_pct     = ('is_night',     'mean'),
    unique_products    = ('product_id',    'nunique'),
    unique_aisles      = ('aisle_id',      'nunique'),
    unique_departments = ('department_id', 'nunique'),
    organic_pct        = ('organic',       'mean'),
    alcohol_flag       = ('alcohol_reord', 'max'),
)

#### post group by 
df_output['hundred_orders'] = df_output['max_order_number'].ge(100).astype(int)
df_output['alcohol_flag'] = df_output['alcohol_flag'].astype(int)              # dont actually need this .... maybe?

df_user_id = df.groupby(['user_id'], as_index=False).agg(
            # order_id
            # eval_set
            # order_number
                orders = ('order_number', 'nunique'),                                                               ### target???
                hundred_orders = ('order_number', lambda x: x[df.loc[x.index, 'order_number'] >= 100].nunique()),   ### target???   # flag if customer has 100+ orders
            # order_dow
                unqiue_days = ('order_dow', 'nunique'),                                                                             # number of distinct days of week a customer ordered on
                #weekend_orders = ('order_number', lambda x: x[df.loc[x.index, 'order_dow'] <= 1].nunique()),
                #weekday_orders = ('order_number', lambda x: x[df.loc[x.index, 'order_dow'] > 1].nunique()),
                weekend_orders_pct = ('order_number', lambda x: x[df.loc[x.index, 'order_dow'] <= 1].nunique() / x.nunique()),
                weekday_orders_pct = ('order_number', lambda x: x[df.loc[x.index, 'order_dow'] > 1].nunique() / x.nunique()),


            # order_hour_of_day
                morning_orders_pct = ('order_number', lambda x: x[df.loc[x.index, 'order_hour_of_day'].between(5,11)].nunique() / x.nunique()),
                afternoon_orders_pct= ('order_number', lambda x: x[df.loc[x.index, 'order_hour_of_day'].between(12,19)].nunique() / x.nunique()),
                night_order_pct = ('order_number', lambda x: x[~df.loc[x.index, 'order_hour_of_day'].between(5,19)].nunique() / x.nunique()),

            # product_id
                unqiue_products = ('product_id', 'nunique'),        ### Perhaps indicative of a taget factor?????
            # add_to_cart_order
            # reordered
            # aisle_id
                unqiue_aisles = ('aisle_id', 'nunique'),            ### Perhaps indicative of a taget factor?????
            # department_id
                unqiue_departments = ('department_id', 'nunique'),  ### Perhaps indicative of a taget factor?????
            # organic
                organic_pct = ('organic', 'mean')#,

            #,**{col: (col, 'mean') for col in dummy_cols}        # Dynamically unpacks as (column, function) tuples instead of a dictionary mapping
            # product_cat
                alcohol_flag = ('reordered', lambda x: x[df.loc[x.index, 'product_cat'] == 'alcohol'].max())
            )

df_user_id

#marker_cats = ['alcohol', 'babies', 'pets', 'international']

In [ ]:
# now make this faster & put it up top
df_minus_firstorder = df[df['order_number']!=0]
reorder_rate = df_minus_firstorder.groupby(['user_id'], as_index=False).agg(
                        reorder_rate = ('reordered', 'mean'),

                        discovery_cart_pct_n5 = 1-('reordered', lambda x: x[df.loc[x.index, 'add_to_cart_order'] <= 5].mean()),
                        discovery_cart_pct_n3 = 1-('reordered', lambda x: x[df.loc[x.index, 'add_to_cart_order'] <= 3].mean()),

                        TARGET_avg_days_since_prior = ('days_since_prior_order', 'mean'),
                        std_days_since_prior = ('days_since_prior_order', 'std'),

                        avg_reorder_size=('order_id', lambda x: df.loc[x.index, 'reordered'].sum() / x.nunique()),

                        same_day_repurchase = ('order_id', lambda x: x[df.loc[x.index, 'days_since_prior_order'] == 0].nunique()),
                        same_week_repurchase = ('order_id', lambda x: x[df.loc[x.index, 'days_since_prior_order'] > 7].nunique()),

                        days_since_over30 = ('order_id', lambda x: x[df.loc[x.index, 'days_since_prior_order'] >= 30].nunique()),
                        days_since_under30 = ('order_id', lambda x: x[df.loc[x.index, 'days_since_prior_order'] < 30].nunique())  
                        )
reorder_rate


In [ ]:
# so if we are clustering (based on previous purchase history??) then what are some factors we might want to consider?
    # number of orders
    # avg time since last order & std dev of time between orders
    # avg number of items per order & std dev



    # categories:
        # weekly purchaser (all purchases 7 days apart)? or always on same day of week (or maybe weekened vs weekday shopers or maybe just percentage split)
        # single product/category customers?


In [ ]:
# INPUTS: cluster on composition, style, and timing features that are roughly invariant to how much someone uses the product
    #
    #
    #
    #
    #



# OUTPUTS: variable used as a downstream outcome is excluded from the inputs; 
    # volume: number of items
    # frequency & legacy???
    #
# Note: related-but-distinct variables may remain but are expected to show partial mechanical correlation, which is disclosed.